# Results: SmolVLA on Neil, task "pick up the green cup"
Physical AI WS, 2026-10-22. Two fine-tuning runs of SmolVLA on the same new dataset, compared: **cold** (starting from
the general model) and **warm** (starting from the team's model that already knows Neil).

### Words used in this notebook

| Word | Meaning |
|---|---|
| **demo** | One recording of a person driving Neil with the gamepad through the whole task, start to finish (about 6 to 20 s). We recorded 50 and used 49. |
| **frame** | One moment of a demo. The recorder runs at **10 Hz**: every 0.1 s it saves the newest image from each camera and the newest motor reading. So a 10 s demo has 100 frames. |
| **data point** | What the model learns from: **one frame** of one demo, with: the 2 camera images of that frame, the 4 motor readings (state) of that frame, the task sentence, and the **next 50 actions** of the demo starting at that frame (5 s). Near the end of a demo there are fewer than 50 actions left; the missing ones are marked as padding and left out of the loss. |
| **batch** | **32 data points** drawn at random. The model sees one batch, computes the loss over its 32 data points, and updates its weights **once**. LeRobot calls one batch a "step". |
| **epoch** | Enough batches to see every training frame once: 4617 frames / 32 = about 144 batches. |
| **Athena job** | A task we submitted to Athena's queue (the Technion GPU cluster). Slurm gives every job a number; the log of the run is named after it. |

### The training runs and their job numbers

| Job | What | Starts from | When |
|---|---|---|---|
| **181719** | The team's earlier run, the **pos2** model: "pick up the cup" (a paper cup), 47 demos | `lerobot/smolvla_base` | 2026-10-14 |
| **267612** | Our **cold** run: "pick up the green cup", 10,000 batches | `lerobot/smolvla_base` (general robot model, never saw Neil) | 2026-10-22 |
| **267613** | Our **warm** run: "pick up the green cup", 5,000 batches | the pos2 model (job 181719) | 2026-10-22 |

Run this notebook with the kernel **Python (.venv-vla, LeRobot)**. It reads only local files: the recordings in `logs/`,
the dataset in `training/data/green_cup/`, the models in `models/` and the training logs in `training/runs/green_cup/`.

In [ ]:
import glob
import json
import re
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "training" else Path.cwd()
LOGS, DATA, RUNS = ROOT / "logs", ROOT / "training/data/green_cup", ROOT / "training/runs/green_cup"
EXCLUDED = ["session_20261022_124142"]
JOBS = {"cold": 267612, "warm": 267613}
pd.set_option("display.width", 160)

## 1. Data used

- **What:** 50 teleoperated demos recorded on 2026-10-22 between 12:41 and 13:18. Neil was driven with the USB gamepad
  (`scripts/gamepad_drive.py --record`) to a green coffee cup, closed the gripper on it and lifted it. The cup and Neil's
  start position changed between demos.
- **Each demo** is one folder `logs/session_<date>_<time>/` with, at **10 steps a second**:
  - `pi/NNNNNN.jpg`: Neil's front camera (Pi Camera, 640x480)
  - `external/NNNNNN.jpg`: the phone above the table (1280x720)
  - `steps.jsonl`: one line per step with the motor readings (`motors`), the gamepad target (`action`) and the buttons pressed
- **Excluded:** 1 demo, `session_20261022_124142`. No button was pressed and Neil never moved: it would teach the model to sit still.

In [ ]:
rows = []
for d in sorted(LOGS.glob("session_20261022_*")):
    steps = [json.loads(l) for l in open(d / "steps.jsonl")]
    rows.append({"session": d.name[-6:], "steps": len(steps), "seconds": len(steps) / 10,
                 "used": d.name not in EXCLUDED})
sess = pd.DataFrame(rows)
eps = pd.read_parquet(DATA / "meta/episodes/chunk-000/file-000.parquet")[["episode_index", "length"]]
print(f"Recorded: {len(sess)} demos, {sess.steps.sum()} steps ({sess.seconds.sum() / 60:.1f} min)")
print(f"Used:     {sess.used.sum()} demos -> dataset: {len(eps)} episodes, {eps.length.sum()} frames "
      f"({eps.length.sum() / 10 / 60:.1f} min) after cutting quiet frames")
print(f"Episode length: {eps.length.min()} to {eps.length.max()} frames, median {eps.length.median():.0f} "
      f"({eps.length.median() / 10:.1f} s)")

fig, ax = plt.subplots(2, 6, figsize=(16, 4.4))
for a, d in zip(ax.flat, sorted(LOGS.glob("session_20261022_*"))[1::4][:12]):
    first = json.loads(open(d / "steps.jsonl").readline())
    a.imshow(cv2.cvtColor(cv2.imread(str(d / first["external"]["file"])), cv2.COLOR_BGR2RGB))
    a.set_title(d.name[-6:], fontsize=9); a.axis("off")
fig.suptitle("Start of 12 demos (phone camera): the cup and Neil's start position vary"); plt.tight_layout(); plt.show()

## 2. From recordings to data points

`training/convert_episodes.py` (the team's converter, unchanged) turned the 49 folders into one LeRobot dataset,
`training/data/green_cup`, with the task sentence **"pick up the green cup"**. One **data point** = one step of one demo:

| Field | What | How it is made |
|---|---|---|
| `observation.images.camera1` | Pi camera, 640x480 RGB | the step's `pi/` image |
| `observation.images.camera2` | phone camera, 640x360 RGB | the step's `external/` image, shrunk from 1280x720 (no crop) |
| `observation.state` (4 numbers) | what the motors **measure** | left wheel speed, right wheel speed, arm angle, gripper angle |
| `action` (4 numbers) | what the gamepad **asked for** | degrees each wheel still has to turn, arm target, gripper target |
| `task` | the sentence | "pick up the green cup" |

Details the converter takes care of:
- **Fixed zero for the arm and gripper.** The motors' `pos_deg` restarts at 0 whenever the Pi program restarts, so the
  converter uses the absolute angle instead: arm = -(abs - 14), gripper = -(abs + 93). The same pose gives the same number in every demo.
- **Signs.** The left wheel, the arm and the gripper motors are mounted reversed, so they are flipped: + always means forward.
- **Wheel action = degrees still to turn**, wrapped to -180..180, not the raw target angle (which jumps from +180 to -180).
- **Quiet frames cut.** Still frames at the start and end of each demo are removed, keeping 1 second on each side.
- **One frame = one recorder tick.** Both cameras stream faster than 10 Hz, so at each tick the recorder takes the
  newest image of each camera. If a camera had not delivered a new image since the last tick (rare, Wi-Fi), the
  same image is used again; `steps.jsonl` marks it with `"new": false`.

The model sees one data point and has to predict the **next 50 actions** (5 seconds) from it.

**How the images are stored and decoded:**
1. The converter reads each frame's JPEG with OpenCV, resizes it (phone: 1280x720 to 640x360, INTER_AREA), and converts
   BGR to RGB.
2. LeRobot then **encodes each camera into one video file** (MP4, AV1 codec, 10 frames per second, quality CRF 30, a
   keyframe every 2 frames): `videos/observation.images.camera1/…/file-000.mp4` and the same for camera2. This makes
   the dataset 83 MB instead of about 2 GB of JPEGs. AV1 is lossy, so the images are very slightly changed.
3. During training, for every data point, LeRobot **decodes the frame from the video** with PyAV at that frame's
   timestamp (it must match within 0.1 ms), giving an RGB image with values 0 to 1.
4. Then the augmentation (section 3) is applied, and inside SmolVLA the image is padded to a square and resized to 512x512.

In [ ]:
d = pd.read_parquet(DATA / "data/chunk-000/file-000.parquet")
state, action = np.stack(d["observation.state"]), np.stack(d["action"])
names = ["left wheel", "right wheel", "arm", "gripper"]
print("One data point (episode 0, frame 30):")
r = d[(d.episode_index == 0) & (d.frame_index == 30)].iloc[0]
print("  state  =", dict(zip(names, np.round(r["observation.state"], 1))))
print("  action =", dict(zip(["left deg to go", "right deg to go", "arm target", "gripper target"], np.round(r["action"], 1))))
print()
print(pd.DataFrame({"state min": state.min(0), "state max": state.max(0), "action min": action.min(0),
                    "action max": action.max(0)}, index=names).round(1))

## 3. Data augmentation (LeRobot's built-in image transforms)

Turned on with `--dataset.image_transforms.enable=true`. The dataset itself is **not changed**: every time a frame is
loaded for training, LeRobot picks **3 of these 6 transforms at random** (`max_num_transforms=3`), in a random order
(`random_order=true`), separately for each camera image. So the model never sees exactly the same image twice.

| Transform | Range |
|---|---|
| brightness | x0.8 to x1.2 |
| contrast | x0.8 to x1.2 |
| saturation | x0.5 to x1.5 |
| hue | -0.05 to +0.05 |
| sharpness | x0.5 to x1.5 |
| affine | rotate -5 to +5 degrees, shift up to 5% |

All of these keep the meaning of the numbers: a brighter or slightly shifted image still needs the same action.
Not used on purpose: **flips** (a mirrored image would swap left and right turns, so the wheel actions would be wrong),
big crops or rotations, grayscale (the cup's color matters).

In [ ]:
import torch
from lerobot.transforms.transforms import ImageTransforms, ImageTransformsConfig

tf = ImageTransforms(ImageTransformsConfig(enable=True, max_num_transforms=3, random_order=True))
torch.manual_seed(0)
d0 = sorted(LOGS.glob("session_20261022_*"))[10]
step = [json.loads(l) for l in open(d0 / "steps.jsonl")][40]
fig, ax = plt.subplots(2, 5, figsize=(16, 5.5))
for row, cam, size in ((0, "pi", (640, 480)), (1, "external", (640, 360))):
    img = cv2.cvtColor(cv2.resize(cv2.imread(str(d0 / step[cam]["file"])), size, interpolation=cv2.INTER_AREA), cv2.COLOR_BGR2RGB)
    x = torch.from_numpy(img).permute(2, 0, 1).float() / 255
    for c in range(5):
        y = x if c == 0 else tf(x)
        ax[row, c].imshow(y.permute(1, 2, 0).clamp(0, 1).numpy()); ax[row, c].axis("off")
        ax[row, c].set_title("original" if c == 0 else f"augmented #{c}", fontsize=9)
fig.suptitle("The same frame as the model sees it in training: original and 4 random augmentations"); plt.tight_layout(); plt.show()

## 4. Other decisions, and what was clipped

| Decision | Value | Why |
|---|---|---|
| Held-out episodes | 10% of the episodes (**5 of 49**), never trained on | to measure `eval_loss`: how well the model does on demos it has not seen |
| Normalization | state and action: (value - mean) / std. Images: not normalized | the model works on standardized numbers. SmolVLA's default |
| Image size inside the model | padded to a square and resized to 512x512 | SmolVLA's default. The 4:3 and 16:9 images keep their shape |
| Padding | state and action padded from 4 to 32 numbers (padded part not in the loss) | SmolVLA's fixed size |
| Frozen parts | the vision-language model (SmolVLM2) is frozen. Only the action expert and the state projection train | about 100M of 450M parameters. Faster, keeps what the base model knows |
| Gradient clipping | gradient norm clipped at 10 | stops single bad batches from making huge updates |
| Seed | 1000 for both runs | the same held-out episodes and the same sample order for cold and warm |

**Clipping on the robot.** This happens only when a trained model drives Neil, never in training. The model outputs
4 numbers per action, and before they are sent to the motors they pass three limits:

1. **The training range.** For each of the 4 action numbers, we know the smallest and the largest value it ever had
   in the data the model was trained on (all frames of all demos used, computed when the dataset was made and saved
   in the model's files as `action.min` / `action.max`). A model output outside that range is cut to the nearest edge.
   Example for the green_cup models: the arm target in all demos was between -19 and 3, so if the model ever outputs
   -30, Neil gets -19. The reason: outside the range the model is guessing about something it never saw a person do.
   The ranges are printed in section 2 (action min / max).
2. **Hard safety caps** in the control panel, the same for every model: a wheel turns at most 30 degrees per command,
   the arm target stays in -25..10 and the gripper target in -80..5.
3. **Deadband:** a wheel command under 3 degrees is sent as 0, so the wheels don't twitch.

## 5. Batches

Training goes **batch by batch**. **Batch size: 32 data points**, the same in both runs and constant for the whole
run (the short test runs before used 8). Each batch is 32 data points drawn at random from the 44 training demos.
For each batch:
1. the model predicts the next 50 actions for each of the 32 data points,
2. the loss is computed and averaged over the 32,
3. the weights are changed **once** (one update per batch).

| | cold | warm |
|---|---|---|
| Training frames (44 demos) | 4617 | 4617 |
| Batches per epoch | 4617 / 32 ≈ 144 | ≈ 144 |
| Batches in the run | 10,000 | 5,000 |
| Epochs (each frame seen ... times) | ≈ 69 | ≈ 35 |
| Time per batch | ≈ 0.5 s (0.33 s on the GPU, 0.16 s loading, decoding and augmenting the images) | same |

The `eval_loss` was measured every 500 batches on the 5 held-out demos.

## 6. Loss function and the loss curves

**Which loss:** SmolVLA's default, a **flow-matching loss**, which is a mean squared error (MSE) between two "velocities":

1. Take the true next 50 actions $a$ (normalized) and random noise $\epsilon$ of the same shape.
2. Pick a random time $t$ between 0 and 1 (from a Beta(1.5, 1) distribution, so mostly near 1) and mix them:
   $x_t = t\,\epsilon + (1-t)\,a$.
3. The network sees the images, state, sentence and $x_t$, and predicts the velocity $v$ that leads from noise to the actions.
4. The target velocity is $u = \epsilon - a$. **Loss = mean of $(v - u)^2$** over the 50 steps and the 4 action values
   (the 28 padded values and any actions past the end of the demo are left out).

**Why this loss fits Neil:**
- Actions are **continuous numbers** (degrees), so a regression loss (MSE) is natural.
- There is often **more than one right move** (turn left or right around the cup, grip now or one step later). Plain MSE
  on the actions would average the options into a wrong "middle" move. Flow matching learns the whole distribution and
  then samples one consistent plan from it (10 denoising steps at run time).
- It predicts **50 steps as one chunk**, so the plan is smooth over 5 seconds.
- It is what `smolvla_base` and pos2 were trained with, so both starting points fit it without any change.

The plot shows the loss against **epochs** (1 epoch = every training frame seen once, about 144 batches). Solid: training loss (on the 44
training episodes). Dots: `eval_loss` (on the 5 held-out episodes).

In [ ]:
TRAIN = re.compile(r"step:(\S+) smpl:\S+ ep:\S+ epch:([\d.]+) loss:([\d.]+) grdn:([\d.]+) lr:([\d.e+-]+) "
                   r"updt_s:([\d.]+) data_s:([\d.]+)")
EVAL = re.compile(r"step (\d+): eval_loss=([\d.]+)")


def parse(job):
    text = (RUNS / f"lina_v2_{job}.log").read_text(errors="ignore")
    t = pd.DataFrame(TRAIN.findall(text), columns=["step", "epoch", "loss", "grad_norm", "lr", "updt_s", "data_s"])
    freq = float(t.step[0])                     # the first line still shows the exact step
    t["step"] = [(i + 1) * freq for i in range(len(t))]
    t = t.astype(float)
    ev = pd.DataFrame([(int(s), float(l)) for s, l in EVAL.findall(text)], columns=["step", "eval_loss"])
    per_step = (t.epoch / t.step).median()      # epochs per batch = 32 / training frames
    ev["epoch"] = ev.step * per_step
    frames = int(re.search(r"num_frames=(\d+)", text).group(1))
    return t, ev, frames


runs = {name: parse(job) for name, job in JOBS.items()}
for name, (t, ev, frames) in runs.items():
    print(f"{name}: {frames} training frames, {t.step.iloc[-1]:.0f} steps = {t.epoch.iloc[-1]:.1f} epochs, "
          f"final train loss {t.loss.iloc[-1]:.3f}, best eval {ev.eval_loss.min():.3f} "
          f"(batch {ev.step[ev.eval_loss.idxmin()]}), final eval {ev.eval_loss.iloc[-1]:.3f}")

fig, ax = plt.subplots(figsize=(11, 5.5))
for (name, (t, ev, _)), color in zip(runs.items(), ("tab:blue", "tab:orange")):
    ax.plot(t.epoch, t.loss, color=color, label=f"{name}: train loss")
    ax.plot(ev.epoch, ev.eval_loss, "o--", color=color, label=f"{name}: eval loss (held-out episodes)")
ax.set(xlabel="epoch", ylabel="loss (flow-matching MSE)", yscale="log", title="Loss per epoch: cold vs warm start")
ax.grid(alpha=.3); ax.legend(); plt.show()

**What the curves show (facts from the logs):**

| | cold | warm |
|---|---|---|
| Training loss, start (batch 100) | 1.163 | 1.078 |
| Training loss, end | 0.129 (batch 10,000, epoch 69) | 0.181 (batch 5,000, epoch 35) |
| Lowest `eval_loss` | 0.733 at batch 500 (epoch 3.5) | 0.779 at batch 500 (epoch 3.5) |
| Last `eval_loss` | 1.441 | 1.037 |
| `eval_loss` at batch 4,000 (epoch 27.7) | 1.059 | 1.038 |

- The training loss goes down in both runs, and reaches each level earlier in the warm run.
- The `eval_loss` (5 held-out demos) is lowest at its first measurement, batch 500, in both runs, and higher at every
  later measurement.
- `eval_loss` was measured every 500 batches only, so nothing is known about batches 0 to 500, and there is no value for
  the starting models (smolvla_base, pos2) before training.

**To look into next (with more GPU time):** `eval_loss` more often in the first batches and for the starting models,
repeated measurements to see how much the numbers vary, training runs with different settings (length, learning rate,
augmentation), and more demos with more variation (cup positions, start poses, scenes).

## 7. Learning rate and optimizer

Both are **SmolVLA's defaults**; only the warm run's peak learning rate was lowered.

- **Optimizer: AdamW** (betas 0.9 / 0.95, eps 1e-8, weight decay 1e-10). After every batch it changes each trainable
  weight a little against its gradient, with a step size that adapts per weight.
- **Learning rate** = the overall size of those changes. It changes after every batch: it rises from about 0 to the peak
  during a short warmup (333 batches cold, 166 warm), then goes down along a cosine curve to 1/40 of the peak at the
  last batch.

| | cold | warm |
|---|---|---|
| Peak learning rate | 1e-4 (default) | 5e-5 (half, so the pos2 weights change more gently) |
| Last learning rate | 2.5e-6 | 1.25e-6 |

In [ ]:
fig, ax = plt.subplots(figsize=(10, 3.8))
for (name, (t, _, _)), color in zip(runs.items(), ("tab:blue", "tab:orange")):
    ax.plot(t.step, t.lr, color=color, label=name)
ax.set(xlabel="batch", ylabel="learning rate", title="Learning rate after every batch"); ax.grid(alpha=.3); ax.legend()
plt.show()

## 8. Warm start: why we could reuse the pos2 model

A warm start only makes sense if the new data **means the same thing** as the data pos2 was trained on: the same inputs
in the same places, and the same numbers on the same scale. What we checked:

| Check | pos2 | green_cup | Same? |
|---|---|---|---|
| Recorder and converter | `gamepad_drive.py --record`, `convert_episodes.py` | same programs | ✅ |
| Action meaning | wheel degrees still to turn, arm and gripper targets | same (the recorder's action names are `left_wheel.pos`, `right_wheel.pos`, `arm.pos`, `grip.pos` in both) | ✅ |
| Zero points and signs | arm abs - 14, gripper abs + 93, left wheel / arm / gripper flipped | same | ✅ |
| Cameras | camera1 = Pi 640x480, camera2 = phone 640x360 | same | ✅ |
| Frame rate | 10 per second | 10 per second | ✅ |
| Chunk | 50 actions | 50 actions (not changed, so the action expert's output keeps its meaning) | ✅ |
| Task sentence | "pick up the cup" | "pick up the green cup" | close, on purpose |
| Scene | paper cup, old lab | green coffee cup, new lab | different, which is the point of the new training |

**The normalization statistics.** The model never sees degrees, only (value - mean) / std. pos2's weights learned pos2's
mean and std. LeRobot replaces them with the new dataset's statistics when a warm start begins, so we compared the two
(`training/stats_compare.py`): if every value of the new data had fitted inside pos2's min..max we would have kept
pos2's mean and std, so every number would mean exactly what it meant before. 6 of the 8 values went slightly outside
(by a few frames each, and the arm state up to 14 where pos2 stopped at 12), so the rule chose the **new data's own
statistics**. They are close to pos2's (table below), so the warm model only had to adjust its scale a little: its
loss was high for the first ~200 batches and then dropped below the cold run's.

In [ ]:
from safetensors.numpy import load_file

pos2 = load_file(str(ROOT / "models/handoff_pick_cup_pos2/pretrained_model/policy_preprocessor_step_5_normalizer_processor.safetensors"))
new = json.load(open(DATA / "meta/stats.json"))
rows = []
for key, labels in (("observation.state", ["left speed", "right speed", "arm", "gripper"]),
                    ("action", ["left deg to go", "right deg to go", "arm target", "gripper target"])):
    for i, lab in enumerate(labels):
        p = {f: float(pos2[f"{key}.{f}"][i]) for f in ("mean", "std", "min", "max")}
        n = {f: float(new[key][f][i]) for f in ("mean", "std", "min", "max")}
        rows.append({"value": f"{key.split('.')[-1]}: {lab}",
                     "pos2 mean±std": f"{p['mean']:.1f} ± {p['std']:.1f}", "new mean±std": f"{n['mean']:.1f} ± {n['std']:.1f}",
                     "pos2 range": f"{p['min']:.0f}..{p['max']:.0f}", "new range": f"{n['min']:.0f}..{n['max']:.0f}",
                     "inside": "yes" if p["min"] <= n["min"] and n["max"] <= p["max"] else "no"})
pd.DataFrame(rows).set_index("value")

## 9. The weights we chose and saved

LeRobot does not pick a "best" model. It saves a checkpoint every N batches (cold: every 2000, warm: every 1000) and
calls the last one `last`. We chose, in each run, **the saved checkpoint with the lowest `eval_loss`**, and kept the
final one next to it. (The lowest `eval_loss` of all was at batch 500, but no checkpoint was saved there.)

In [ ]:
saved = {"warm": [1000, 2000, 3000, 4000, 5000], "cold": [2000, 4000, 6000, 8000, 10000]}
rows = []
for name, (t, ev, _) in runs.items():
    e = dict(zip(ev.step, ev.eval_loss))
    best = min(saved[name], key=lambda b: e[b])
    for b in saved[name]:
        rows.append({"run": name, "checkpoint (batch)": b, "epoch": round(b * 32 / 4617, 1), "eval_loss": e[b],
                     "kept": "chosen" if b == best else ("final" if b == saved[name][-1] else "")})
pd.DataFrame(rows).set_index(["run", "checkpoint (batch)"])

The 4 kept models are in `green_cup_2026.10.22/2_warm_start_weights` (`warm_batch1000_chosen`, `warm_batch5000_final`)
and `green_cup_2026.10.22/3_cold_start_weights` (`cold_batch2000_chosen`, `cold_batch10000_final`), each as a
`pretrained_model` folder (the 7 files needed to run it). All other checkpoints, with the training states needed to
continue training, are in `Neil-LabAI/models/`. On the robot, each model is run with its own spec
(`specs/green_cup_warm.json`, `specs/green_cup_cold.json`; they differ only in the action range used for clipping).